# Tutorial: a knowledge graph for a scientific study

This notebook builds the knowledge graph of a small study with pubmed-stream and walks through what it contains. A study has:

- a **topic**, which names its knowledge graph folder `kg_<topic>`;
- **PubMed searches** that collect its literature;
- optionally, **key articles** whose evidence neighborhood (similar, citing and cited papers) belongs in it.

`create_kg` downloads the full text of the searches, fetches the neighborhoods, and links everything in an [Obsidian](https://obsidian.md) vault: one Markdown note per paper, MeSH concept and collection. Open the folder in Obsidian as a vault to explore it in the graph view.

The example study asks how the **gut microbiome relates to frailty**. The notebook queries NCBI, so it needs internet access; it takes about two minutes. Setting the `NCBI_API_KEY` environment variable makes it about three times faster.

## 1. Setup

Use the repository version when the notebook runs from a clone, and work in a scratch folder so the tutorial's downloads stay out of the way.

In [1]:
import os
import sys
from pathlib import Path

# Run from a clone: import the package from the repository root
REPO = Path.cwd().parent
if (REPO / "pubmed_stream").is_dir():
    sys.path.insert(0, str(REPO))

import pubmed_stream
from pubmed_stream import create_kg, load_study

WORKSPACE = Path("kg_tutorial_workspace")
WORKSPACE.mkdir(exist_ok=True)
os.chdir(WORKSPACE)  # the graph and downloads are created here
print("pubmed-stream", pubmed_stream.__version__)

pubmed-stream 0.1.1


## 2. Start the study with one search

`create_kg` takes the topic and the searches. Each search is downloaded into `publications/<query>/`, then the knowledge graph is built in `kg_gut_microbiome_frailty/`. PubMed field tags such as `[Title]` keep the search focused.

In [2]:
TOPIC = "gut microbiome frailty"

result = create_kg(
    TOPIC,
    queries=['"gut microbiota"[Title] AND frailty[Title/Abstract]'],
    max_results=15,
)
print(result)


Found 15 PMC IDs for '"gut microbiota"[Title] AND frailty[Title/Abstract]' (total PubMed results: 66)
Output directory: publications\gut_microbiota_Title_AND_frailty_Title_Abstract



2 articles were not available in PMC full-text


Progress: 15/15 processed (OK:13 | FAIL:2 | SKIP:0)

Download Summary
Keyword:           "gut microbiota"[Title] AND frailty[Title/Abstract]
Total found:       66
Requested:         15
[OK] Successful:   13
[FAIL] Failed:     2
  - Unavailable:   2
  - Errors:        0
[SKIP] Skipped:    0
Duration:          6.8s
Output directory:  publications\gut_microbiota_Title_AND_frailty_Title_Abstract



Knowledge graph: gut microbiome frailty
Folder:            kg_tutorial_workspace\kg_gut_microbiome_frailty
Study searches:    1 (1 run now, 13 new articles)
Study seeds:       0 (0 fetched now)
Obsidian vault
Vault:             kg_tutorial_workspace\kg_gut_microbiome_frailty
Sources:           1
Papers:            13 (13 full text, 0 stubs)
MeSH concepts:     23
Collections:       1
Notes:             38 created, 0 updated, 0 unchanged
PubMed records:    13 fetched
  - Without MeSH:  8 (offline, or not yet indexed in PubMed)


The summary has two parts. The first describes the **study**: how many searches and seeds it has, and what ran this time. The second describes the **notes** that were written. *Without MeSH* counts papers PubMed has not indexed with MeSH terms yet, which is common for papers from the last year.

## 3. What is in the knowledge graph folder

In [3]:
KG = result.kg_dir
for folder in ["papers", "concepts", "collections"]:
    notes = sorted((KG / folder).glob("*.md"))
    print(f"{folder + '/':13} {len(notes):3} notes, e.g. {notes[0].name if notes else '-'}")
print(f"{'home note':13} {(KG / 'PubMed Stream.md').name}")

papers/        13 notes, e.g. PMID41345866.md
concepts/      23 notes, e.g. Aging.md
collections/    1 notes, e.g. Search - gut microbiota(Title) AND frailty(Title-Abstract).md
home note     PubMed Stream.md


- `papers/` has one note per paper, named by PubMed ID.
- `concepts/` has one note per MeSH heading, the topics that connect papers.
- `collections/` has one note per search or neighborhood.
- `PubMed Stream.md` is the home note to start from in Obsidian.

## 4. Read a paper note

Each paper note starts with Obsidian properties, followed by a generated block with the abstract, MeSH concepts and citation links. Links such as `[[PMID31319564|...]]` are what Obsidian draws as edges in the graph view. Here is the note of the paper with the most major MeSH topics, shortened:

In [4]:
import re

def show_note(path, properties=8, body_lines=32, width=150):
    """Print a note's first properties and the start of its body, long lines cut."""
    _, frontmatter, body = path.read_text(encoding="utf-8").split("---", 2)
    props = frontmatter.strip().splitlines()
    lines = ["---"] + props[:properties] + (["..."] if len(props) > properties else []) + ["---"]
    body = [l if len(l) <= width else l[:width] + " ..." for l in body.strip("\n").splitlines()]
    lines += body[:body_lines] + (["..."] if len(body) > body_lines else [])
    print("\n".join(lines))

# The paper with the most major MeSH topics
papers = sorted((KG / "papers").glob("*.md"), key=lambda p: -p.read_text(encoding="utf-8").count("(major topic)"))
show_note(papers[0])

---
type: "paper"
title: "Gut Microbiota-Targeted Nutrition for Healthy Aging: Mechanistic Roles of Polyphenols and Dietary Fiber in Geroscience"
aliases:
  - "Gut Microbiota-Targeted Nutrition for Healthy Aging: Mechanistic Roles of Polyphenols and Dietary Fiber in Geroscience"
pmid: "42588100"
pmcid: "PMC13468338"
doi: "10.3390/nu18152478"
year: 2026
...
---
%% pubmed-stream:start - generated; edits inside this block are overwritten %%
# Gut Microbiota-Targeted Nutrition for Healthy Aging: Mechanistic Roles of Polyphenols and Dietary Fiber in Geroscience

**Authors:** Kryczyk-Poprawa Agata, Rząsa-Duran Elżbieta, Varga János Tamás, Lehoczki Andrea, Zábó Virág, Fazekas-Pongor Vince, Major Dávid, Csípő Tam ...
**Published:** *Nutrients* · 2026 · [PubMed](https://pubmed.ncbi.nlm.nih.gov/42588100/) · [PMC](https://pmc.ncbi.nlm.nih.gov/articles/PMC13468338/) ·  ...
**Publication type:** Journal Article, Review
**Collections:** [[Search - gut microbiota(Title) AND frailty(Title-Abstract)]]


## 5. Read a concept note and the home note

A concept note lists the study's papers indexed with that MeSH heading, newest first. The home note gives an overview of the whole graph.

In [5]:
concepts = sorted((KG / "concepts").glob("*.md"), key=lambda p: -p.read_text(encoding="utf-8").count("- [["))
show_note(concepts[0], properties=3, body_lines=16)

---
type: "concept"
vocabulary: "MeSH"
mesh_ui: "D000069196"
...
---
%% pubmed-stream:start - generated; edits inside this block are overwritten %%
# Gastrointestinal Microbiome

MeSH descriptor D000069196 · [MeSH Browser](https://meshb.nlm.nih.gov/record/ui?ui=D000069196)

## Papers (5)

- [[PMID41683327|Composition of the Gut Microbiota in Older Adults Residing in a Nursing Home and Its Association... (2026)]] (major topic)
- [[PMID42420265|Gut microbiota associates with frailty in older women (2026)]] (major topic)
- [[PMID42588100|Gut Microbiota-Targeted Nutrition for Healthy Aging: Mechanistic Roles of Polyphenols and Dietary... (2026)]] (major topic)
- [[PMID41361501|Association between a dietary index for gut microbiota and frailty in U.S. adults aged 40 and old... (2025)]] (major topic)
- [[PMID41345866|Associations of dietary intake for gut microbiota with frailty risk and mortality in older adults (2025)]] (major topic)
%% pubmed-stream:end %%

## Notes


In [6]:
show_note(KG / "PubMed Stream.md", properties=0, body_lines=26)

---
...
---
%% pubmed-stream:start - generated; edits inside this block are overwritten %%
# PubMed Stream

Knowledge graph of the papers downloaded with pubmed-stream. Open the graph view to explore it.

| Notes | Count |
|---|---|
| Papers with full text | 13 |
| Stub papers | 0 |
| MeSH concepts | 23 |
| Collections | 1 |

## Collections

- [[Search - gut microbiota(Title) AND frailty(Title-Abstract)]] (13 papers)

## Most connected concepts

- [[Gastrointestinal Microbiome]] (5 papers)
- [[Frailty]] (3 papers)
- [[Aging]] (2 papers)
- [[Bacteria]] (2 papers)
- [[Diet]] (2 papers)
- [[Frail Elderly]] (2 papers)
- [[Nutrition Surveys]] (2 papers)
- [[Cardiovascular Diseases]] (1 paper)
...


## 6. Add a key article's evidence neighborhood

A search finds papers by their words. A **seed** adds a key article's neighborhood: papers PubMed rates as similar, papers citing it, and papers it cites. Neighbors that were not downloaded become short *stub* notes with their PubMed details. Pass `download_neighbors=True` to download their full text too.

The seed here is a 2019 review, *Gut Microbiota, Muscle Mass and Function in Aging: A Focus on Physical Frailty and Sarcopenia* (PMID 31319564). The same topic adds it to the same graph.

In [7]:
result = create_kg(TOPIC, seeds=["31319564"], max_per_relation=20)
print(result)


Knowledge graph: gut microbiome frailty
Folder:            kg_tutorial_workspace\kg_gut_microbiome_frailty
Study searches:    1 (0 run now, 0 new articles)
Study seeds:       1 (1 fetched now)
Obsidian vault
Vault:             kg_tutorial_workspace\kg_gut_microbiome_frailty
Sources:           2
Papers:            72 (13 full text, 59 stubs)
MeSH concepts:     204
Collections:       2
Notes:             241 created, 21 updated, 17 unchanged
PubMed records:    8 fetched
  - Without MeSH:  16 (offline, or not yet indexed in PubMed)


In [8]:
hood = result.neighborhoods[0]
print("Seed:", hood.seed.title)
for relation in ["similar", "cited_by", "references"]:
    print(f"{relation:11} {len(getattr(hood, relation)):3} fetched of {hood.totals[relation]}")
print()
show_note(KG / "collections" / "Neighborhood - PMID 31319564.md", properties=3, body_lines=16)

Seed: Gut Microbiota, Muscle Mass and Function in Aging: A Focus on Physical Frailty and Sarcopenia.
similar      20 fetched of 384
cited_by     20 fetched of 239
references   20 fetched of 139

---
type: "collection"
kind: "neighborhood"
seed: "PMID31319564"
...
---
%% pubmed-stream:start - generated; edits inside this block are overwritten %%
# Evidence neighborhood of PMID 31319564

60 papers, 1 with full text.

**Seed:** [[PMID31319564|Gut Microbiota, Muscle Mass and Function in Aging: A Focus on Physical Frailty and Sarcopenia. (2019)]]

## Top MeSH concepts

- [[Gastrointestinal Microbiome]] (40)
- [[Sarcopenia]] (26)
- [[Muscle, Skeletal]] (24)
- [[Aging]] (15)
- [[Frailty]] (11)
- [[Exercise]] (7)
- [[Muscle Strength]] (7)
...


## 7. Follow the citations

The seed's paper note now links to the papers it cites and to the papers citing it. In Obsidian, these are the edges around the seed in the graph view, and the *Backlinks* panel shows them from the other side.

In [9]:
seed_note = (KG / "papers" / "PMID31319564.md").read_text(encoding="utf-8")
for heading in ["## Cites", "## Cited by", "## Similar articles"]:
    section = seed_note.split(heading, 1)[1].split("\n## ", 1)[0] if heading in seed_note else ""
    links = re.findall(r"^- (\[\[.*)$", section, flags=re.M)
    print(f"{heading[3:]} ({len(links)} links), first three:")
    for link in links[:3]:
        print("  ", link[:130])

Cites (20 links), first three:
   [[PMID31061952|Effect of a Multistrain Probiotic on Cognitive Function and Risk of Falls in Patients With Cirrho... (2019)]]
   [[PMID31039010|Gut bacteria are critical for optimal muscle function: a potential link with glucose homeostasis. (2019)]]
   [[PMID31027674|Pathway from gait speed to incidence of disability and mortality in older adults: A mediating rol... (2019)]]
Cited by (20 links), first three:
   [[PMID42137883|Association Between Dietary Index for Gut Microbiota and Sarcopenic Obesity in Middle-Aged and El... (2026)]]
   [[PMID42380078|Bidirectional Relationship and Shared Mechanisms Between Sarcopenia and Osteoporosis: An Observat... (2026)]]
   [[PMID42332518|Dietary index for gut microbiota: A new frontier in sarcopenia prevention. (2026)]]
Similar articles (20 links), first three:
   [[PMID29189738|Aging Gut Microbiota at the Cross-Road between Nutrition, Physical Frailty, and Sarcopenia: Is Th... (2017)]] · sco
   [[PMID31226349|Gu

## 8. Your own notes survive rebuilds

Generated content sits between two `%% pubmed-stream %%` markers. Anything written outside them, such as reading notes under `## Notes`, and any properties or tags added in Obsidian, are kept when the graph is rebuilt.

In [10]:
note = KG / "papers" / "PMID31319564.md"
note.write_text(note.read_text(encoding="utf-8") + "Key review for the introduction; see Table 2.\n", encoding="utf-8")

result = create_kg(TOPIC)  # no new searches: rebuild the notes only
print("Notes updated:", result.vault.updated, "| unchanged:", result.vault.unchanged)
print("My note kept:", "Key review for the introduction" in note.read_text(encoding="utf-8"))

Notes updated: 0 | unchanged: 279
My note kept: True


## 9. Keep the study current

The graph remembers its study, so later sessions need only the topic. `refresh=True` runs every search and neighborhood again and adds papers published since; articles already downloaded are skipped.

In [11]:
print(load_study(KG))
# Later, for example once a month:
# create_kg(TOPIC, refresh=True)

{'topic': 'gut microbiome frailty', 'queries': ['"gut microbiota"[Title] AND frailty[Title/Abstract]'], 'seeds': ['31319564']}


## 10. Explore it in Obsidian

1. Open Obsidian and choose **Open folder as vault**.
2. Select `examples/kg_tutorial_workspace/kg_gut_microbiome_frailty`.
3. Open `PubMed Stream` and then the **graph view**. Retracted papers are red, MeSH concepts teal, collections purple, reviews orange and stubs grey.
4. In the graph view's filters, `path:papers` shows the citation network alone, and `tag:#pubmed/review` finds the reviews.

## The same study from the command line

```bash
pubmed-stream kg "gut microbiome frailty" \
    -q '"gut microbiota"[Title] AND frailty[Title/Abstract]' --max-results 15 \
    --seed 31319564 --max-per-relation 20
pubmed-stream kg "gut microbiome frailty" --refresh
```

See the *Obsidian Knowledge Graph* section of the README for every option.